# UC4 Aim Assist — YOLOv8 Enemy Detector Training

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Upload `dataset_yolo.zip` to your **Google Drive** root folder first
3. Then run cells in order

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), 'NO GPU! Go to Runtime > Change runtime type > T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'CUDA: {torch.version.cuda}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Install ultralytics ───────────────────────────────────────────────
!pip install ultralytics -q
from ultralytics import YOLO
print('ultralytics ready')

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────────────────
# This will ask you to sign in to Google — use the same account
# where you uploaded dataset_yolo.zip
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── Cell 4: Find and extract dataset ─────────────────────────────────────────
import os, zipfile

# Looks for dataset_yolo.zip in Drive root and common subfolders
search_paths = [
    '/content/drive/MyDrive/dataset_yolo.zip',
    '/content/drive/MyDrive/UC4/dataset_yolo.zip',
    '/content/drive/MyDrive/uc4_aim_assist/dataset_yolo.zip',
]
zip_path = next((p for p in search_paths if os.path.exists(p)), None)

if zip_path is None:
    # List Drive root so you can find where the file is
    print('dataset_yolo.zip not found in common locations. Files in Drive root:')
    for f in os.listdir('/content/drive/MyDrive'):
        print(' ', f)
    raise FileNotFoundError('Upload dataset_yolo.zip to Google Drive and re-run this cell.')

print(f'Found: {zip_path}')
print('Extracting... (this takes 2-3 minutes for 2.8 GB)')

os.makedirs('/content/dataset', exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/dataset')

for split in ['train', 'val', 'test']:
    img_dir = f'/content/dataset/images/{split}'
    n = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    print(f'  {split}: {n} images')

In [ ]:
# ── Cell 5: Fix data.yaml paths for Colab ────────────────────────────────────
import yaml

data_yaml_path = '/content/dataset/data.yaml'

data = {
    'path':  '/content/dataset',
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc':    1,
    'names': ['enemy'],
}
with open(data_yaml_path, 'w') as f:
    yaml.dump(data, f)

print('data.yaml ready:')
print(open(data_yaml_path).read())

In [ ]:
# ── Cell 6: Train YOLOv8n ────────────────────────────────────────────────────
# Expected time: ~30-45 min on T4 GPU
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data          = data_yaml_path,
    epochs        = 150,
    batch         = 16,
    imgsz         = 640,
    device        = 0,
    workers       = 4,
    patience      = 30,
    project       = '/content/runs',
    name          = 'enemy_detector',
    # Augmentation
    hsv_h         = 0.015,
    hsv_s         = 0.7,
    hsv_v         = 0.4,
    fliplr        = 0.5,
    mosaic        = 1.0,
    mixup         = 0.1,
    degrees       = 5.0,
    translate     = 0.1,
    scale         = 0.5,
    # Hyperparams
    lr0           = 0.01,
    lrf           = 0.001,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    box           = 7.5,
    cls           = 0.5,
    verbose       = True,
)

best = '/content/runs/enemy_detector/weights/best.pt'
print(f'\nTraining complete!')
print(f'mAP50   : {results.results_dict.get("metrics/mAP50(B)", "N/A")}')
print(f'mAP50-95: {results.results_dict.get("metrics/mAP50-95(B)", "N/A")}')
print(f'Weights : {best}')

In [ ]:
# ── Cell 7: Save best.pt to Google Drive (survives session end) ───────────────
import shutil

drive_out = '/content/drive/MyDrive/uc4_best.pt'
shutil.copy('/content/runs/enemy_detector/weights/best.pt', drive_out)
print(f'Saved to Google Drive: {drive_out}')
print('You can now download it from drive.google.com')

In [ ]:
# ── Cell 8: Also direct-download to your PC ───────────────────────────────────
from google.colab import files
files.download('/content/runs/enemy_detector/weights/best.pt')
print('Downloading best.pt to your PC...')
print('Place it at:')
print('  C:\\Systeme\\uc4_aim_assist\\models\\training\\enemy_detector\\weights\\best.pt')